# Snowflake DCM Projects (PrPr) - Quickstart 1 (Workspaces)

⚠️ Note: This project will only work if your account is already enrolled in the Private Preview of Snowflake DCM Projects.

---

## Project Files

This is a demo DCM Project for you to explore some of the core capabilities. 

Explore the content of the manifest file and the various definition files:
- The Manifest contains different target profiles and templating configurations
- The definition files contain various DEFINE statements for new entities, as well as GRANTS, ATTACH DMF and examples for jinja templating
- the macros folder is optional and can store global jinja macros to be used across different definition files

**Tip:** Use the split-screen to keep manifest and definition files left and this setup file on the right

#

## Role Setup

**Step 1 (optional, but recommended):** Create a dedicated developer role to manage DCM Projects on a DEV environment



In [ ]:
%%sql -r dataframe_1
use role ACCOUNTADMIN;

create role if not exists DCM_DEVELOPER;
set USER_NAME = (select current_user());
grant role DCM_DEVELOPER to user identifier($USER_NAME);

**Step 2:** Grant permissions to create new account infrastructure through DCM deployments
(depending on your needs and project purpose)

In [ ]:
%%sql -r dataframe_2
grant CREATE WAREHOUSE on account to role DCM_DEVELOPER;
grant CREATE ROLE on account to role DCM_DEVELOPER;
grant CREATE DATABASE on account to role DCM_DEVELOPER;
grant EXECUTE MANAGED TASK on account to role DCM_DEVELOPER;
grant EXECUTE TASK on account to role DCM_DEVELOPER;

grant MANAGE GRANTS on account to role DCM_DEVELOPER;

**Step 3:**  If you want to deefine and test data quality expectations you also need:

In [ ]:
%%sql -r dataframe_3
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role DCM_DEVELOPER;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role DCM_DEVELOPER;
grant database role SNOWFLAKE.DATA_METRIC_USER to role DCM_DEVELOPER;
grant execute data metric function on account to role DCM_DEVELOPER;

**Step 4 (optional):** In case you don't have a warehouse available or want to run the deployments on a dedicates warehouse

(DCM commands are mostly meta-data changes, which makes them very lightweight. Only few commands actually require a warehouse and XS is sufficient)

In [ ]:
%%sql -r dataframe_22
create warehouse if not exists DCM_WH
with
    warehouse_size = 'XSMALL'
    auto_suspend = 300
    comment = 'For Quickstart Demo of DCM Projects'
;  

## DCM Project Object Setup
Create a Snowflake object inside a schema that executes and stores all deployments of your project definitions 

In [ ]:
%%sql -r dataframe_4
use role DCM_DEVELOPER;
  
create database if not exists DCM_DEMO;
create schema if not exists DCM_DEMO.PROJECTS;

create or replace dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_DEV
    comment = 'for testing DCM Projects Quickstarts'
;

## Dry-run a deployment using the PLAN command

**1. In the DCM control panel above the tabs, select the project `DCM_Project_Quickstart_1`**

- See how the `DCM_DEV` target is already pre-selected in top right as it is set as default in the manifest
- click on the target profile to see that it uses the previously created DCM_PROJECT_DEV object and the `DEV` templating configuration
- override the templating value for `user` with your own user-name
- optional: open the manifest file to see what other values this configuration contains

**2. Execute a "Plan" to the DCM_DEV target environment**
- click 'Plan' and wait for the definitions to render, compile and dry-run the statements
    
**3. Review the plan result** (if the plan output does not show as a new tab, click again on the blue "Plan" button)

- If you get an error, then use the message to debug it 
- Review all the changes that were planned. Because non of these objects exist yet, there are only CREATE statements
- expand one of the Dynamic Table sections to see more details of the definitions 
- Try the AI Summary button

**3. Review the plan output files**

- In the file explorer you can see a new  `out` folder was created above `sources`
- open it to see the `plan_output` which contains the rendered jinja templating for all definition files.
- open the `jinja_demo.sql` file from the plan output next to the `jinja_demo.sql` file from the sources to see how the Jinja was rendered

### SQL commands
Alternatively to the Workspace UI you can also manually execute the SQL commands.

The json output structure will be less readable for humans - but very useful for coding agents to validate their work. 

See below what SQL command the Workspace is constructing based on the manifest target profile and then executes on your behalf:

In [ ]:
%%sql -r dataframe_5
execute dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_DEV
    PLAN
    using configuration DEV (user => 'YOUR_USER_NAME')
    from
        'snow://workspace/USER$.PUBLIC."snowflake_dcm_projects"/versions/live/Quickstarts/DCM_Project_Quickstart_1'
;

## Deploy the Project Definitions

If the plan result was successfull and all planned changes are correct, you can now safely `Deploy` those changes.

- In the top-right corner of you PLAN results tab - click "Deploy".
- Optional: You can add a Deployment-alias which you can think of as a "commit name" which you can see later in the deployment history of your project to understand what this deployment was for.
- DCM will create all the objects and attach grants and expectations using the owner-role of the project object.
- Once the deployment completed successfully, you can refresh the Database Explorer on the left to see the DCM_DEMO_1_DEV database and all created objects inside.

### SQL commands

Just like PLAN  you can also run the DEPLOY as a SQL command. 

While the json output format is very easy to interpret for Cortex Code, we can also flatten it to make it more readable for humans.

In [ ]:
%%sql -r dataframe_6
execute dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_DEV 
    DEPLOY
    using configuration DEV (user => 'YOUR_USER_NAME')
    from
        'snow://workspace/USER$.PUBLIC."snowflake_dcm_projects"/versions/live/Quickstarts/DCM_Project_Quickstart_1'

    -- flatten json output to table
    ->> with JSON_INPUT as (
            select parse_json($1) as OPERATIONS from $1) 
        select
            f.INDEX ::number as INDEX,
            f.value:operationType ::string as OPERATION_TYPE,
            coalesce( f.value:objectDomain, f.value:association) ::string as OBJECT_TYPE,
            coalesce( f.value:objectName, f.value:target:objectName) ::string as OBJECT_NAME,
            coalesce( f.value:details:properties, concat(f.value:subject:objectPrivilege,' on ',f.value:subject:objectName)) ::string as PROPERTIES,
            f.value:details:columns ::string as COLUMNS,
            f.value ::string as VALUE
        from
            JSON_INPUT,
            LATERAL FLATTEN(input => JSON_INPUT.OPERATIONS)f
        order by
            INDEX
;

## Executing SQL post-scripts

Independently of DCM Projects you can also execute separate sql files to create objects that are not yet supported by DCM (using CREATE IF NOT EXISTS or CREATE OR REPLACE) or manually trigger Task runs, etc. 

We will use it to insert sample data to the Quickstart raw tables.

The command runs a predefied `post_hook.sql`file using **jinja templating** to match our DCM configuration variable.

In [ ]:
%%sql -r dataframe_7
execute immediate
    from 'snow://workspace/USER$.PUBLIC."snowflake_dcm_projects"/versions/live/Quickstarts/DCM_Project_Quickstart_1/SQL_post_scripts/insert_sample_data.sql'
    using (env_suffix => '_DEV')
;

---
# DCM Projects for Pipelines

You can leverage additional DCM capabilities when creating and deploying transformation pipeline that leverage Dynamic Tables and Data Metric Functions. 

## REFRESH Command

Refresh all Dynamic Tables defined inside the project to process new sample data that was inserted by running the following SQL command:

In [ ]:
%%sql -r dataframe_9
execute dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_DEV 
    REFRESH ALL

    -- optional flow operator to format the json output as table
    ->> select
            f.value:data_timestamp ::string as TIMESTAMP_IN_MS,
            f.value:dt_name ::string as DT_NAME,
            f.value:statistics ::string as RESULT,
            f.value:refreshed_dt_count ::number as REFRESHED_DTS
        from
            $1 as REFRESH_RESULT,
            lateral flatten(input => parse_json(REFRESH_RESULT."result"):refreshed_tables) f
;

## TEST Command

Now that the sample data was loaded and processed throughout the transformation pipeline, you can test all Data Quality Expectations that were set on table-objects inside the project.

Run the command below:

In [ ]:
%%sql -r dataframe_8
execute dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_DEV 
    TEST ALL

    -- optional flow operator to format the json output as table
    ->> select
            f.value:table_name ::string as TABLE_NAME,
            f.value:metric_database ::string as METRIC_DATABASE,
            f.value:metric_schema ::string as METRIC_SCHEMA,
            f.value:metric_name ::string as METRIC_NAME,
            f.value:expectation_name ::string as EXPECTATION_NAME,
            f.value:expectation_expression ::string as EXPECTATION_EXPRESSION,
            case 
                when f.value:expectation_violated ::boolean = 'FALSE' then '✅ MET'
                when f.value:expectation_violated ::boolean = 'TRUE' then '🚫 FAILED'
                end as EXPECTATION_RESULT,
            f.value:value ::number as VALUE
        from
            $1 as TEST_RESULTS,
            lateral flatten(input => parse_json(TEST_RESULTS."message"):expectations) f     
;

## PREVIEW Command

If you are making changes to pipeline definitions then you can preview a data-sample even before deploying the change.

Preview can be run for a defined table / dynamic table / view.

For example you can change the definition of the ENRICHED_ORDER_DETAILS Dynamic Table.

Then run the PREVIEW command for the downstream V_DASHBOARD_KPI_SUMMARY View to see how the change impacts the data of the downstream View.

In [ ]:
%%sql -r dataframe_10
execute dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_DEV
    PREVIEW DCM_PROJECT_DEV.SERVE.V_DASHBOARD_KPI_SUMMARY
    using configuration DEV (user => 'YOUR_USER_NAME')
    from
        'snow://workspace/USER$.PUBLIC."snowflake_dcm_projects"/versions/live/Quickstarts/DCM_Project_Quickstart_1'
;

---
# DCM Deployment Observability

The DCM Project object stores artifacts from all deployments to provide a full audit trail.
For each deployment it contains:
- a metadata file with timestamp, query ID, executing role and templating values
- a snapshot of the manifest.yml at the time of the deployment
- a snapshot of all definition files at the time of the deployment
- the rendered jinja output of all definitions

Open your `Horizon Catalog` and navigate to DCM_DEMO / PROJECTS / DCM Projects / DCM_PROJECT_DEV to see the deployment history and the changes that were made during each deployment.

Switch to the Database Explorer and go to DCM_DEMO.PROJECTS to see all DCM Project objects in that schema.

Or use the `SHOW` command to review what DCM Projects exist in our database and which role owns them.

(requires READ privilege on the DCM Project(s).)

In [ ]:
%%sql -r dataframe_11
show dcm projects in schema DCM_DEMO.PROJECTS;

The `DESCRIBE` command also contains the timestamp of the most recent deployment.

(the source and deployment paths are only visible to user with MONITOR privileges)

In [ ]:
%%sql -r dataframe_12
describe dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_DEV;

---
# Plan and Deploy to a Production Environment

Now that you have deployed the Quickstart project definitions to a DEV environment, let's see how we can leverage the manifest configurations to deploy the same objects to a PROD instance on the same account.

#
**Step 1:** Create dedicated service role to execute DCM Project deployments to PROD

In [ ]:
%%sql -r dataframe_16
use role ACCOUNTADMIN;

create role if not exists DCM_PROD_DEPLOYER;

set USER_NAME = (select current_user());
grant role DCM_PROD_DEPLOYER to user identifier($USER_NAME);   -- replace with your CI/CD service-user

grant USAGE on warehouse DCM_WH to role DCM_PROD_DEPLOYER;
grant USAGE on database DCM_DEMO to role DCM_PROD_DEPLOYER;
grant USAGE on schema DCM_DEMO.PROJECTS to role DCM_PROD_DEPLOYER;
grant CREATE DCM PROJECT on schema DCM_DEMO.PROJECTS to role DCM_PROD_DEPLOYER;

**Step 2:** Grant permissions to create new account infrastructure through DCM deployments

In [ ]:
%%sql -r dataframe_17
grant CREATE WAREHOUSE on account to role DCM_PROD_DEPLOYER;
grant CREATE ROLE on account to role DCM_PROD_DEPLOYER;
grant CREATE DATABASE on account to role DCM_PROD_DEPLOYER;
grant EXECUTE MANAGED TASK on account to role DCM_PROD_DEPLOYER;
grant EXECUTE TASK on account to role DCM_PROD_DEPLOYER;

grant MANAGE GRANTS on account to role DCM_PROD_DEPLOYER;

If you want to set and test data quality expectations:

In [ ]:
%%sql -r dataframe_18
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role DCM_PROD_DEPLOYER;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role DCM_PROD_DEPLOYER;
grant database role SNOWFLAKE.DATA_METRIC_USER to role DCM_PROD_DEPLOYER;
grant execute data metric function on account to role DCM_PROD_DEPLOYER;

## PROD PROJECT OBJECT SETUP

Each target environment needs its own project - no matter if they are on the same account or not

In [ ]:
%%sql -r dataframe_19
use role DCM_PROD_DEPLOYER;

create or replace dcm project DCM_DEMO.PROJECTS.DCM_PROJECT_PROD
    comment = 'Simulating deployment to PROD with a service-user'
;

- Change the target profile for the DCM Project from DCM_DEV to DCM_PROD_US
- Run PLAN to see what changes would be deployed to the Prod US environment (in this demo all environments are on the same Snowflake account. But they might also be on different accounts)
- Carefully review the PLAN result
- Run DEPLOY
- Refresh the Database Explorer to see the newly created DCM_DEMO_1 database


⚠️ Notice how you need a dedicated DCM PROJECT object for each target environment so that these deployed instances can co-exist on one account. 

Developers might have the DCM_DEVELOPER role to deploy changes to DEV. But they might not hold the DCM_PROD_DEPLOYER role, so they can not directly deploy to the production environment.

---
Now you can **create your own new project** by selecting "+ Add new" on the file-explorer on the left and then "DCM Project"

If you want to run DCM commands in your **local IDE**, then checkout the `setup_cli.md` file in this project.